# ESMFold2 (MLX/Metal) memory vs. token count — find *your* machine's token limit

`run_esmfold2` can die with
`[metal::malloc] Attempting to allocate N bytes which is greater than the maximum allowed buffer size of M bytes`
(process abort, exit 134) when a bait+prey complex is too long. Memory grows with the
number of tokens (residues) roughly **super-quadratically (≈ n^2.2 measured)** (pair representations), so there is a hard
ceiling that depends on your Mac's RAM, on the Metal single-buffer cap, and on
`esmfold2.num_diffusion_samples`.

This notebook:
1. reads your hardware limits from MLX,
2. folds synthetic complexes of increasing length **in a subprocess** (so the Metal abort at the ceiling is survivable) and records peak GPU memory,
3. fits memory(n) and extracts the exact single-buffer coefficient from the crash message,
4. recommends a **max token limit** (with a safety margin), shows how it changes with `num_diffusion_samples`, and counts how many of your pairs it would exclude.

**Run it with nothing else heavy open** — the ramp ends by deliberately crashing the worker once. Takes a while (model load ~25 GB + folds up to the ceiling).
Memory does not depend on sequence content, so random sequences are used.

In [ ]:
import json, math, re, subprocess, sys, time
from pathlib import Path
import numpy as np, yaml
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
cfg = yaml.safe_load((ROOT / "config.yaml").read_text())
local = ROOT / "config.local.yaml"
if local.exists():
    cfg["esmfold2"].update((yaml.safe_load(local.read_text()) or {}).get("esmfold2", {}))
EF = cfg["esmfold2"]

# ── knobs ─────────────────────────────────────────────────────────────────
N_SAMPLES      = int(EF["num_diffusion_samples"])    # buffer size scales with this -> keep equal to the pipeline's
PROBE_LENGTHS  = [100, 200, 300, 450, 600, 750, 900, 1050, 1200, 1300, 1400, 1500, 1600, 1800, 2000]
PROBE_LOOPS    = int(EF["num_loops"])           # measured: peak memory is flat in loops/steps after the first warm-up fold,
PROBE_STEPS    = int(EF["num_sampling_steps"])  # but a low-setting probe under-reads by ~15%, so use the pipeline values
BAIT_FRACTION  = 0.5   # chain split of the synthetic complex (2 chains, like bait+prey)
SAFETY_MARGIN  = 0.10  # recommended limit = predicted ceiling * (1 - margin)
RERUN          = True  # False: reuse the last probe from disk
# Crash message seen in logs/folding/run_esmfold2/A1L4W5__Q0WV90.log (1535 tokens, 3 samples).
# Used for the buffer model if this probe never reaches a crash. Set to [] to ignore.
PRIOR_BUFFER_POINTS = [(1535, 28953292800, 3)]    # (tokens, bytes requested, num_diffusion_samples)

PROBE_DIR = ROOT / "results" / "esmfold2_memory_probe"
PROBE_DIR.mkdir(parents=True, exist_ok=True)
print("root:", ROOT, "| samples:", N_SAMPLES)

## 1. Python with `mlx_lm` + `esm` (the pipeline's esmfold2 env)

In [ ]:
import glob, os
cands = [sys.executable] + sorted(glob.glob(str(ROOT / ".snakemake/conda/*/bin/python")))
PY = None
for c in cands:
    if subprocess.run([c, "-c", "import mlx_lm, esm, mlx.core"], capture_output=True).returncode == 0:
        PY = c; break
assert PY, "No python with mlx_lm + esm found: run the workflow once (--use-conda) or install envs/esmfold2.yaml"
print("worker python:", PY)

## 2. Hardware limits reported by Metal

In [ ]:
code = ("import json, mlx.core as mx\n"
        "f = getattr(mx, 'device_info', None) or mx.metal.device_info\n"
        "print(json.dumps({k: (v if isinstance(v, (int, float, str)) else str(v)) for k, v in f().items()}))")
HW = json.loads(subprocess.run([PY, "-c", code], capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1])
GB = 1024**3
MAX_BUFFER = HW["max_buffer_length"]
WORKING_SET = HW.get("max_recommended_working_set_size", HW.get("memory_size"))
print({k: (f"{v/GB:.1f} GiB" if isinstance(v, int) and v > 1e8 else v) for k, v in HW.items()})

## 3. Probe worker
Loads the model once, then folds synthetic complexes of ascending length, appending one JSON line per length
(`peak_bytes` = Metal peak during that fold, `base_bytes` = memory right after loading the weights).
The ramp ends when Metal aborts the process; the driver reads the crash message from the log.

In [ ]:
WORKER = PROBE_DIR / "probe_worker.py"
WORKER.write_text(r"""
import argparse, json, random, time
import mlx.core as mx
from mlx_lm.models.esmfold2 import ESMFold2Model
from esm.models.esmfold2 import ESMFold2InputBuilder, ProteinInput, StructurePredictionInput

ap = argparse.ArgumentParser()
ap.add_argument("--checkpoint"); ap.add_argument("--out")
ap.add_argument("--lengths", type=int, nargs="+")
ap.add_argument("--samples", type=int); ap.add_argument("--loops", type=int); ap.add_argument("--steps", type=int)
ap.add_argument("--bait-fraction", type=float, default=0.5)
a = ap.parse_args()

peak = getattr(mx, "get_peak_memory", None) or mx.metal.get_peak_memory
active = getattr(mx, "get_active_memory", None) or mx.metal.get_active_memory
reset = getattr(mx, "reset_peak_memory", None) or mx.metal.reset_peak_memory
clear = getattr(mx, "clear_cache", None) or mx.metal.clear_cache
emit = lambda d: (open(a.out, "a").write(json.dumps(d) + "\n"))

model = ESMFold2Model.from_pretrained(a.checkpoint)
builder = ESMFold2InputBuilder()
base = active()
emit({"event": "loaded", "base_bytes": base})
rng = random.Random(0)
for n in a.lengths:
    n_a = max(1, int(n * a.bait_fraction))
    seqs = ["".join(rng.choices("ACDEFGHIKLMNPQRSTVWY", k=k)) for k in (n_a, n - n_a)]
    spi = StructurePredictionInput(sequences=[ProteinInput(id=c, sequence=s) for c, s in zip("AB", seqs)])
    clear(); reset(); t = time.time()
    res = builder.fold(model, spi, num_loops=a.loops, num_sampling_steps=a.steps,
                       num_diffusion_samples=a.samples, seed=1)
    got = len(res) if isinstance(res, list) else 1
    emit({"event": "ok" if got == a.samples else "short", "n": n, "peak_bytes": peak(),
          "base_bytes": base, "seconds": time.time() - t, "samples_returned": got})
    if got != a.samples:   # pipeline treats silently-dropped samples as failure
        break
""")
print("wrote", WORKER)

## 4. Run the ramp (survives the Metal abort)

In [ ]:
def drain(path, seen):
    if not path.exists(): return seen
    lines = path.read_text().splitlines()
    for l in lines[seen:]:
        d = json.loads(l)
        if d["event"] == "loaded": print(f"model loaded: {d['base_bytes']/GB:.1f} GiB resident")
        else: print(f"n={d['n']:>5} {d['event']:<5} peak={d['peak_bytes']/GB:6.1f} GiB  {d['seconds']:.0f}s")
    return len(lines)

def run_probe(lengths, tag, samples=N_SAMPLES):
    out, log = PROBE_DIR / f"{tag}.jsonl", PROBE_DIR / f"{tag}.log"
    out.unlink(missing_ok=True)
    cmd = [PY, str(WORKER), "--checkpoint", EF["checkpoint"], "--out", str(out), "--samples", str(samples),
           "--loops", str(PROBE_LOOPS), "--steps", str(PROBE_STEPS), "--bait-fraction", str(BAIT_FRACTION),
           "--lengths", *map(str, lengths)]
    seen = 0
    with open(log, "w") as lf:
        p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT)
        while p.poll() is None:
            seen = drain(out, seen); time.sleep(3)
    drain(out, seen)
    events = [json.loads(l) for l in out.read_text().splitlines()] if out.exists() else []
    text = log.read_text()
    m = re.search(r"Attempting to allocate (\d+) bytes which is greater than the maximum allowed buffer size of (\d+)", text)
    ok_n = [e["n"] for e in events if e["event"] == "ok"]
    failed_n = next((n for n in lengths if n not in ok_n), None)
    return dict(events=events, returncode=p.returncode, failed_n=failed_n,
                alloc=int(m.group(1)) if m else None, cap=int(m.group(2)) if m else None, log=str(log))

SUMMARY = PROBE_DIR / "ramp_summary.json"
if RERUN or not SUMMARY.exists():
    ramp = run_probe(PROBE_LENGTHS, "ramp")
    SUMMARY.write_text(json.dumps(ramp))
ramp = json.loads(SUMMARY.read_text())
print("\nreturncode:", ramp["returncode"], "| first failing length:", ramp["failed_n"],
      "| crash alloc:", ramp["alloc"], "| log:", ramp["log"])

## 5. Fit the growth curves

* **Total peak memory** above the resident weights: `extra(n) = A·n^p` (log-log least squares; p comes out ≈ 2.2 on an M4 Max, steeper than quadratic).
* **Largest single buffer**: the Metal error reports the requested bytes `X` at the failing length `n_f`; the biggest tensor is
  pair-shaped, so `X ≈ k·n²` with `k = X / n_f²` (and it scales with the number of diffusion samples). Plausibility check on the earlier
  crash: 28 953 292 800 / 1535² = **12 288 B per token²** = 3 samples × 4096 B — an exact integer, consistent with this model.

In [ ]:
ok = [e for e in ramp["events"] if e["event"] == "ok"]
n_ok = np.array([e["n"] for e in ok], float)
extra = np.array([e["peak_bytes"] - e["base_bytes"] for e in ok], float)
base = next((e["base_bytes"] for e in ramp["events"] if e["event"] == "loaded"), 0)

if len(ok) >= 3:
    # power law extra = A * n^p (measured p ~ 2.2 for 1 sample on an M4 Max: steeper than quadratic)
    p_exp, logA = np.polyfit(np.log(n_ok), np.log(extra), 1)
    A = math.exp(logA)
    pred = A * n_ok ** p_exp
    r2 = 1 - ((extra - pred) ** 2).sum() / ((extra - extra.mean()) ** 2).sum()
    print(f"extra(n) = {A:.3e} * n^{p_exp:.2f}   R2={r2:.4f}")
    coef = (A, p_exp)
else:
    coef = None; print("fewer than 3 successful points -> only the buffer model is available")

# single-buffer coefficient (bytes per token^2 per diffusion sample)
if ramp["alloc"] and ramp["failed_n"]:
    k_per_sample = ramp["alloc"] / ramp["failed_n"] ** 2 / N_SAMPLES
    k_src = f"this probe's crash at n={ramp['failed_n']}"
    if ramp["cap"]: MAX_BUFFER = ramp["cap"]
elif PRIOR_BUFFER_POINTS:
    n0, x0, s0 = PRIOR_BUFFER_POINTS[0]
    k_per_sample = x0 / n0 ** 2 / s0
    k_src = f"prior crash log (n={n0}, samples={s0})"
else:
    k_per_sample = None; k_src = "unavailable"
print(f"buffer coefficient: {k_per_sample and round(k_per_sample, 1)} B/token^2/sample  ({k_src}); Metal max buffer {MAX_BUFFER/GB:.2f} GiB")

## 6. Predicted ceilings → recommended max tokens

In [ ]:
def n_buffer_limit(samples):
    if not k_per_sample: return math.inf
    return math.floor(math.sqrt(MAX_BUFFER / (k_per_sample * samples)))

def n_workingset_limit():
    if coef is None: return math.inf
    A, p_exp = coef
    return math.floor(((WORKING_SET - base) / A) ** (1 / p_exp))

def recommend(samples):
    raw = min(n_buffer_limit(samples), n_workingset_limit() if samples == N_SAMPLES else math.inf)
    return raw, int(raw * (1 - SAFETY_MARGIN)) // 10 * 10

raw, REC = recommend(N_SAMPLES)
print(f"single-buffer ceiling : {n_buffer_limit(N_SAMPLES)} tokens")
print(f"working-set ceiling   : {n_workingset_limit()} tokens  (budget {(WORKING_SET-base)/GB:.1f} GiB above weights)")
print(f"=> hard ceiling {raw}, recommended max_tokens = {REC}  (margin {SAFETY_MARGIN:.0%}, samples={N_SAMPLES})\n")
print("sensitivity to num_diffusion_samples (buffer limit only; working-set model is for the probed sample count):")
for s in (1, 2, 3, 5):
    print(f"  samples={s}: ceiling {n_buffer_limit(s)} -> recommended {int(n_buffer_limit(s)*(1-SAFETY_MARGIN))//10*10}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
nn = np.linspace(50, max(PROBE_LENGTHS) * 1.1, 200)
if coef is not None:
    ax[0].plot(nn, (base + coef[0] * nn ** coef[1]) / GB, label="fit")
    ax[0].scatter(n_ok, (base + extra) / GB, c="k", label="measured peak", zorder=3)
ax[0].axhline(WORKING_SET / GB, c="r", ls="--", label="recommended working set")
ax[0].set(xlabel="tokens", ylabel="peak GiB", title="Total Metal memory"); ax[0].legend()
if k_per_sample:
    ax[1].plot(nn, k_per_sample * N_SAMPLES * nn**2 / GB, label=f"largest buffer ({N_SAMPLES} samples)")
    ax[1].axhline(MAX_BUFFER / GB, c="r", ls="--", label="Metal max buffer")
ax[1].set(xlabel="tokens", ylabel="GiB", title="Largest single allocation")
for a_ in ax: a_.axvline(REC, c="g", ls=":", label="recommended max"); a_.legend()
plt.tight_layout(); plt.show()

## 7. Validate the recommendation (optional but advised)
Folds one complex at exactly the recommended length with the **pipeline's** settings for loops/steps. If it survives, the limit is safe;
if not, increase `SAFETY_MARGIN` and re-run section 6.

In [ ]:
VALIDATE = False
if VALIDATE:
    v = run_probe([REC], "validate")
    print("PASS" if v["failed_n"] is None and v["returncode"] == 0 else f"FAIL: {v['log']}")

## 8. How many pairs would the limit exclude?

In [ ]:
import collections
try: from yaml import CSafeLoader as L
except ImportError: from yaml import SafeLoader as L
lens = {}
for f in sorted((ROOT / "configs").glob("*__*.yaml")):
    d = yaml.load(f.read_text(), Loader=L)
    lens[d["name"]] = sum(len(s["sequence"]) * 1 for s in d["sequences"])
arr = np.array(list(lens.values()))
print(f"{len(arr)} pair specs; tokens: median {np.median(arr):.0f}, p95 {np.percentile(arr,95):.0f}, max {arr.max()}")
over = [k for k, v in lens.items() if v > REC]
print(f"{len(over)} pairs ({len(over)/max(len(arr),1):.1%}) exceed max_tokens={REC}")
plt.hist(arr, bins=60); plt.axvline(REC, c="g"); plt.xlabel("tokens per pair"); plt.show()

## 9. Use it

In [ ]:
print(f"""Add to config.local.yaml (per-machine, gitignored):

esmfold2:
  max_tokens: {REC}      # measured with {N_SAMPLES} diffusion samples; recompute if you change num_diffusion_samples
""")
(PROBE_DIR / "recommendation.json").write_text(json.dumps(
    dict(max_tokens=REC, ceiling=raw, samples=N_SAMPLES, margin=SAFETY_MARGIN, k_per_sample=k_per_sample)))